In [1]:
# backend/
#   app/
#     main.py                     # FastAPI app entry point
#     api/
#       routes_games.py           # REST endpoints (create game, action, get state)
#       ws_games.py               # WebSocket endpoint for realtime updates
#     schemas/                    # Pydantic models (contracts)
#       card_defs.py              # CardDef, PlayDef (loaded from JSON/YAML)
#       actions.py                # Client->server action models
#       responses.py              # Server->client response models (DTOs)
#     services/
#       game_service.py           # Orchestrates: load state, apply engine, save, broadcast
#       card_catalog.py           # Loads + validates cards from files
#     engine/                     # Pure game logic (no web, no DB)
#       state.py                  # dataclasses: GameState, PlayerState, etc.
#       rules.py                  # apply_action(), validation, turn rules
#       effects/
#         registry.py             # effect_name -> function mapping
#         draw.py
#         rent.py
#         steal.py
#         payment.py              # your payment logic lives here
#     db/
#       models.py                 # SQLAlchemy models (Game row, Move row)
#       session.py                # DB session / engine
#       repo.py                   # DB helper functions (get_game, save_game, log_move)
#   cards/
#     base/                       # official card definitions (JSON)
#     custom/                     # your custom cards / mod cards (JSON)
#   tests/
#     test_payment.py
#     test_draw.py
#     test_rent.py
#   pyproject.toml (or requirements.txt)


In [2]:
## Pydantic Models for Cards and Raw Cards

from pydantic import BaseModel, Field, ConfigDict
from typing import Any, Dict, List, Optional, Literal

class RawEffect(BaseModel):
    type: str
    model_config = ConfigDict(extra="allow")  # keep other keys

class RawCard(BaseModel):
    id: str
    name: str
    type: str                 # your JSON uses "type"
    count: int
    bank_value: int

    colors: Optional[List[str]] = None
    property_group: Optional[str] = None
    effect: Optional[RawEffect] = None
    rent_by_count: Optional[List[int]] = None

    bankable: Optional[bool] = None
    image_url: Optional[str] = None
    restrictions: Optional[Dict[str, Any]] = None
    note: Optional[str] = None

    model_config = ConfigDict(extra="allow")  # keep any future fields

class PlayDef(BaseModel):
    effect: str
    params: Dict[str, Any] = Field(default_factory=dict)

class CardDef(BaseModel):
    id: str
    name: str
    kind: Literal["money","property","property_wild","rent","action"]
    copies: int
    money_value: int
    colors: Optional[List[str]] = None
    rent_by_count: Optional[List[int]] = None 
    play: Optional[PlayDef] = None
    meta: Dict[str, Any] = Field(default_factory=dict)


In [3]:
## Convert RawCard to CardDef

#Convert RawCard to CardDef
def raw_to_carddef(raw: RawCard) -> Optional[CardDef]:
    kind_map = {
        "money": "money",
        "property": "property",
        "property_wild": "property_wild",
        "rent": "rent",
        "action": "action"
    }
    
    kind = kind_map.get(raw.type)
    if not kind:
        return None  # Unknown type
    
    play_def = None
    if raw.effect:
        play_def = PlayDef(effect=raw.effect.type, params={k: v for k, v in raw.effect.model_dump().items() if k != "type"})
    
    card_def = CardDef(
        id=raw.id,
        name=raw.name,
        kind=kind,
        copies=raw.count,
        money_value=raw.bank_value,
        rent_by_count=raw.rent_by_count,
        colors=raw.colors,
        play=play_def,
        meta={
            "property_group": raw.property_group,
            "bankable": raw.bankable,
            "image_url": raw.image_url,
            "restrictions": raw.restrictions,
            "note": raw.note
        }
    )
    
    return card_def

In [4]:
## Load all cards

from pathlib import Path
import json
from typing import Dict, List

def load_card_defs_from_dir(cards_dir: str) -> Dict[str, CardDef]:
    catalog: Dict[str,CardDef] = {}

    for path in Path(cards_dir).glob("*.json"):
        raw_data = json.loads(path.read_text(encoding="utf-8"))

        # Normalize to a list
        if isinstance(raw_data, list):
            raw_cards = [RawCard.model_validate(item) for item in raw_data]
        else:
            raw_cards = [RawCard.model_validate(raw_data)]

        for rc in raw_cards:
            cd = raw_to_carddef(rc)
            if cd is None:
                continue
            if cd.id in catalog:
                raise ValueError(f"Duplicate card ID found: {cd.id}")
            catalog[cd.id] = cd

    return catalog

In [5]:
## Build Rent Table from catalog (This will be done once per game)
import os

def build_rent_table(catalog: Dict[str, CardDef]) -> Dict[str, List[int]]:
    rent_table: Dict[str, List[int]] = {}
    for card_def in catalog.values():
        if card_def.kind == "property" and card_def.meta.get("property_group") and card_def.rent_by_count:
            # All properties in a group should have same rent_by_count
            rent_table[card_def.meta["property_group"]] = card_def.rent_by_count
    return rent_table

# Pathing here:
base_path = os.getcwd()

all_cards_json_path = os.path.join(base_path, "all_cards.json")

all_cards_json = json.load(open(all_cards_json_path, "r"))
cards_path = os.path.join(base_path,"backend" ,"cards", "base")

#Usage
catalog = load_card_defs_from_dir(cards_path)

RENT_TABLE = build_rent_table(catalog)

In [6]:
## GameState and PlayerState and DeckState:

from typing import Dict, List, Optional
from pydantic import BaseModel, Field


class DeckState(BaseModel):
    draw_pile: List[str] = Field(default_factory = list)
    discard_pile: List[str] = Field(default_factory = list)

class PlayerState(BaseModel):
    id: str
    hand: List[str] = Field(default_factory=list)        # card ids
    bank: List[str] = Field(default_factory=list)        # card ids
    properties: Dict[str, List[str]] = Field(default_factory=dict)  # color -> card ids
    buildings: Dict[str, List[str]] = Field(default_factory=dict)  # color -> building card ids

class GameState(BaseModel):
    id: str
    players: Dict[str, PlayerState]
    deck: DeckState
    current_player_id: Optional[str] = None
    turn_number: int = 1
    actions_taken: int = 0

In [7]:
## Build Deck Function

import random
from typing import Dict, List

def build_deck(catalog: Dict[str, CardDef], seed = 42) -> List[str]:
    deck: List[str] = []

    #catalog is key: card id, value: CardDef
    for cd in catalog.values():
        deck.extend([cd.id] * cd.copies)
    
    #Shuffle deck with seed
    random.seed(seed)
    random.shuffle(deck)

    #Check if deck is 110 cards long

    if len(deck) != 106: #110 - 4 helper cards
        
        raise ValueError(f"Deck size is {len(deck)}, expected 110.")
    
    return deck


#Test this function out

catalog = load_card_defs_from_dir(cards_path)
deck = build_deck(catalog, seed=123)


In [9]:
## Create New Game:
import uuid
from typing import Dict, List
#from .engine.state import GameState, PlayerState, DeckState #TODO: we will add this later
#from .services.card_catalog import build_deck #TODO: we will also add this later

STARTING_HAND_SIZE = 5

def create_new_game(player_ids: List[str], 
                    catalog: Dict[str, CardDef]) -> GameState:
    # 1) Build and shuffle deck
    draw_pile = build_deck(catalog)
    deck = DeckState(draw_pile=draw_pile, discard_pile=[])

    # 2) Create players
    players = {pid: PlayerState(id=pid) for pid in player_ids}

    # 3) Deal starting hands
    for _ in range(STARTING_HAND_SIZE):
        for pid in player_ids:
            if not deck.draw_pile:
                raise ValueError("Deck is empty while dealing starting hands.")
            card_id = deck.draw_pile.pop(0)
            players[pid].hand.append(card_id)

    # 4) Create game state
    return GameState(
        id=str(uuid.uuid4()),
        players=players,
        deck=deck,
        current_player_id=player_ids[0] if player_ids else None,
        turn_number=1,
    )

In [ ]:
# play_action
#  └── create pending if counterable
#  └── else → apply_action_effect(...)

# respond_to_pending
#  └── if accept → apply_action_effect(...)


def apply_action_effect(
    state: GameState,
    catalog: Dict[str, CardDef],
    *,
    player_id: str,
    card_id: str,
    # rent/action params
    rent_color: Optional[str] = None,
    double_rent_ids: Optional[List[str]] = None,
    target_player_id: Optional[str] = None,
    steal_card_id: Optional[str] = None,
    give_card_id: Optional[str] = None,
    steal_color: Optional[str] = None,
    # if action is already discarded outside (pending path)
    already_discarded: bool = False,
) -> Dict[str, Any]:
    actor = state.players[player_id]
    card = catalog[card_id]

    # discard the action card here only if not already done
    if not already_discarded:
        discard_action_cards(state, actor, [card_id] + (double_rent_ids or []))

    # RENT
    if card.kind == "rent":
        amount = charge_rent_amount(
            state=state,
            catalog=catalog,
            player_id=player_id,
            rent_card_id=card_id,
            color=rent_color,
            double_rent_ids=double_rent_ids,
        )
        return {
            "status": "ok",
            "response_type": "payment_required",
            "state": state,
            "payment_request": {
                "request_id": f"pay_{uuid.uuid4().hex}",
                "receiver_id": player_id,
                "targets": [{"player_id": target_player_id, "amount": amount}],
            },
            "log": {"action": "rent", "player_id": player_id, "amount": amount},
        }

    # ACTIONS
    if card.kind == "action":
        effect = card.play.effect if card.play else None

        if effect == "draw_cards":
            amount = int(card.play.params.get("amount", 0))
            draw_cards(state, player_id, amount)
            return {
                "status": "ok",
                "response_type": "action_resolved",
                "state": state,
                "log": {"action": "draw_cards", "player_id": player_id, "amount": amount},
            }

        if effect in {"steal_full_set", "steal_property", "swap_property"}:
            result = process_property_manipulation(
                state=state,
                catalog=catalog,
                actor_id=player_id,
                action_card_id=card_id,
                target_player_id=target_player_id,
                steal_card_id=steal_card_id,
                give_card_id=give_card_id,
                steal_color=steal_color,
            )
            return {
                "status": "ok",
                "response_type": "action_resolved",
                "state": state,
                "log": result,
            }

        if effect == "charge_players":
            amount = int(card.play.params.get("amount", 0))
            return {
                "status": "ok",
                "response_type": "payment_required",
                "state": state,
                "payment_request": {
                    "request_id": f"pay_{uuid.uuid4().hex}",
                    "receiver_id": player_id,
                    "targets": [{"player_id": target_player_id, "amount": amount}],
                },
                "log": {"action": "charge_players", "player_id": player_id, "amount": amount},
            }

    raise ValueError("Unsupported action effect.")
